# Optunaによるパラメータのオートチューニング

In [ ]:
!pip install -qq optuna kaggle-environments

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.7/721.7 kB 14.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 944.3/944.3 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.3/96.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 840.2/840.2 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.4/111.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.4/267.4 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 73.5 M

In [ ]:
"""Colabで実行する作業員パラメータ探索コード。"""

import importlib
import statistics

import optuna
from kaggle_environments import make

import main as strategy


# Colab上の最新のmain.pyを読み込む。
strategy = importlib.reload(strategy)

N_TRIALS = 100
MATCH_COUNT = 20
EVALUATION_SEEDS = list(range(MATCH_COUNT))


def objective(trial):
    """作業員設定ごとの平均得点を返す。"""
    strategy.StrategyConfig.MAX_HIRES_PER_TURN = trial.suggest_int(
        "MAX_HIRES_PER_TURN",
        1,
        3,
    )
    strategy.StrategyConfig.INITIAL_TARGET_HANDS = trial.suggest_int(
        "INITIAL_TARGET_HANDS",
        4,
        8,
    )
    strategy.StrategyConfig.MIN_LARGE_FARM_HANDS = trial.suggest_int(
        "MIN_LARGE_FARM_HANDS",
        7,
        10,
    )
    strategy.StrategyConfig.MAX_HANDS = trial.suggest_int(
        "MAX_HANDS",
        8,
        10,
    )
    strategy.StrategyConfig.PASS_RATE_TO_DECREASE = trial.suggest_float(
        "PASS_RATE_TO_DECREASE",
        0.10,
        0.30,
        step=0.01,
    )
    strategy.StrategyConfig.PASS_RATE_TO_INCREASE = trial.suggest_float(
        "PASS_RATE_TO_INCREASE",
        0.00,
        0.10,
        step=0.01,
    )

    rewards = []

    for episode_seed in EVALUATION_SEEDS:
        strategy.hire_controller = strategy.HireController()

        env = make(
            "kaggriculture",
            configuration={
                "episodeSteps": 720,
                "seed": episode_seed,
            },
            debug=True,
        )

        env.run([strategy.agent, strategy.agent])

        for state in env.steps[-1]:
            rewards.append(float(state.reward))

    return statistics.fmean(rewards)


study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    show_progress_bar=True,
)

print("\n===== 最高平均得点 =====")
print(study.best_value)

print("\n===== main.pyへ手動設定する値 =====")

for name, value in study.best_params.items():
    print(f"{name} = {value}")

[I 2026-09-07 11:36:29,526] A new study created in memory with name: no-name-890c396c-575a-48d4-a8b4-3a875482feb3


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-09-07 11:39:27,996] Trial 0 finished with value: 64932.8 and parameters: {'MAX_HIRES_PER_TURN': 2, 'INITIAL_TARGET_HANDS': 8, 'MIN_LARGE_FARM_HANDS': 9, 'MAX_HANDS': 9, 'PASS_RATE_TO_DECREASE': 0.13, 'PASS_RATE_TO_INCREASE': 0.01}. Best is trial 0 with value: 64932.8.
[I 2026-09-07 11:42:19,363] Trial 1 finished with value: 59258.35 and parameters: {'MAX_HIRES_PER_TURN': 1, 'INITIAL_TARGET_HANDS': 8, 'MIN_LARGE_FARM_HANDS': 9, 'MAX_HANDS': 10, 'PASS_RATE_TO_DECREASE': 0.1, 'PASS_RATE_TO_INCREASE': 0.1}. Best is trial 0 with value: 64932.8.
[I 2026-09-07 11:45:08,151] Trial 2 finished with value: 47589.025 and parameters: {'MAX_HIRES_PER_TURN': 3, 'INITIAL_TARGET_HANDS': 5, 'MIN_LARGE_FARM_HANDS': 7, 'MAX_HANDS': 8, 'PASS_RATE_TO_DECREASE': 0.16, 'PASS_RATE_TO_INCREASE': 0.05}. Best is trial 0 with value: 64932.8.
[I 2026-09-07 11:48:02,709] Trial 3 finished with value: 61662.325 and parameters: {'MAX_HIRES_PER_TURN': 2, 'INITIAL_TARGET_HANDS': 5, 'MIN_LARGE_FARM_HANDS': 9, 'MAX